# Import

In [ ]:
import polars as pl
from bs4 import BeautifulSoup
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

# Config

In [ ]:
# start with bronze alpha ingestion, then move to silver
ALPHA_BRONZE = Path("../data/bronze/alpha")
SILVER_DIR = Path("../data/silver/alpha")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

# Methods

In [ ]:
def get_clean_text(soup):
    """Extract text from known containers."""
    # identified keywords
    targets = ["post-message", "post_content"]
    
    container = soup.find(class_=targets)
    
    if not container:
        return None
    
    return container.get_text(separator="\n", strip=True)

In [ ]:
print(f"Starting text extraction for Alpha: {ALPHA_BRONZE}")

folders = sorted([f for f in ALPHA_BRONZE.iterdir() if f.is_dir()])
stats = {"success": 0, "failed": 0}

for folder in tqdm(folders, desc="Alpha Patches"):
    patch_id = folder.name
    # Recursive search, if the HTMLs are deeply nested
    html_files = list(folder.rglob("*.html")) + list(folder.rglob("*.HTML"))
    
    if not html_files:
        continue
        
    source = html_files[0]
    
    try:
        with open(source, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f, "lxml")
            
        patch_text = get_clean_text(soup)
        
        if patch_text:
            # save as polars dataframe
            df = pl.DataFrame({
                "patch_version": [f"alpha_{patch_id}"],
                "raw_text": [patch_text],
                "source_file": [source.name]
            })
            
            out_path = SILVER_DIR / f"patch_{patch_id.replace('.', '-')}.parquet"
            df.write_parquet(out_path)
            stats["success"] += 1
        else:
            stats["failed"] += 1
            
    except Exception as e:
        print(f"Fehler bei {patch_id}: {e}")

print(f"\nDone. Alpha patches successful: {stats['success']}. Failed: {stats['failed']}")